# 「猜猜看多少錢」專案 - 第四天：深度學習與大型語言模型

## 專案背景

本週的核心目標是打造一個「商品定價預測模型」：
- **輸入**：從 Amazon 爬取的商品文字描述
- **輸出**：預測該商品的價格

## 本週學習路線

| 天數 | 主題 |
|------|------|
| DAY 1 | 資料蒐集（Data Curation） |
| DAY 2 | 資料前處理（Data Pre-processing） |
| DAY 3 | 評估指標、基準線、傳統機器學習 |
| **DAY 4** | **深度學習與大型語言模型（今天！）** |
| DAY 5 | Fine-tuning 前沿模型 |

## 今天的學習目標

從傳統機器學習出發，逐步進化到神經網路，最後測試目前最強的大型語言模型（LLMs）在這個任務上的表現。

學習順序：**人類基準線 → 人工神經網路（ANN）→ 前沿 LLM 模型比較**

In [1]:
# 匯入所需套件
# - dotenv：載入 .env 中的 API 金鑰
# - huggingface_hub：登入 HuggingFace 以下載資料集
# - pricer.evaluator：Day 3 建立的統一評估函式
# - litellm：統一呼叫各家 LLM API 的包裝層
# - pricer.items：Item 資料結構，封裝商品描述與價格
# - torch / torch.nn / torch.optim：PyTorch 深度學習框架
# - HashingVectorizer：文字向量化（詞袋模型）
# - CosineAnnealingLR：學習率調整策略

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR

In [2]:
# 環境設定與 HuggingFace 登入
# LITE_MODE = True 時使用小型資料集（開發/測試用）
# LITE_MODE = False 時使用完整資料集（正式訓練用）
# hf_token 從 .env 檔案讀取，用於下載私有資料集

LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [3]:
# 從 HuggingFace Hub 載入資料集
# Day 1 爬取、Day 2 前處理後的資料已上傳至 HuggingFace
# Item.from_hub() 會自動分割為 train / val / test 三份

username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 800,000 training items, 10,000 validation items, 10,000 test items


---

# 第一部分：人類基準線（Human Baseline）

在進入人工神經網路之前，先建立「人類預測」的基準線。

**概念**：如果連人類都無法準確猜出商品價格，那機器的表現很難超越人類直覺；反過來，若 AI 能超越人類，就代表模型確實學到了有用的知識。

**做法**：
1. 將測試集的前 100 筆商品描述輸出成 CSV（`human_in.csv`）
2. 人工填入預測價格後另存為 `human_out.csv`
3. 用相同的 `evaluate()` 函式評估人類的預測結果

In [4]:
# 輸出測試集前 100 筆至 CSV 供人工填寫預測價格
# human_in.csv 格式：[商品描述, 0]
# 人工填完後另存為 human_out.csv，將 0 替換成猜測的價格

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [5]:
# 讀取人工填寫完成的預測結果
# human_out.csv 由人工在 human_in.csv 基礎上填寫價格後產生

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [6]:
# 將人類預測值包裝成與其他模型相同介面的函式
# evaluate() 會呼叫 pricer(item)，因此需要統一格式

def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [7]:
# 快速驗證：印出第一筆商品的人類預測值與實際價格，確認資料讀取正確

human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")

Human predicted 120.0 for an item that actually costs 219.0


In [8]:
# 正式評估人類基準線（僅評估前 100 筆，與人工填寫的數量一致）
# evaluate() 會計算 RMSE、MAE 等指標，作為後續模型比較的基準

evaluate(human_pricer, test, size=100)

  0%|          | 0/100 [00:00<?, ?it/s]

$99 $184 $12 $15 $18 $10 $119 $135 $6 $270 $643 $329 $15 $26 $24 $18 $29 $25 $25 $53 $35 $126 $25 $127 $273 $398 $55 $6 $101 $51 $30 $5 $35 $9 $10 $419 $25 $11 $186 $33 $161 $51 $23 $155 $150 $4 $31 $18 $115 $82 $25 $111 $410 $75 $67 $34 $8 $10 $122 $28 $116 $17 $19 $60 $599 $60 $160 $355 $75 $34 $17 $2 $70 $76 $41 $9 $226 $5 $5 $4 $0 $7 $5 $74 $7 $10 $68 $74 $5 $3 $17 $45 $5 $16 $0 $153 $2 $122 $150 $355 

---

# 第二部分：人工神經網路（Artificial Neural Network）

## 從詞袋模型到深度學習

Day 3 使用的是傳統機器學習（如 Ridge Regression、Random Forest），這些方法無法捕捉文字之間的非線性關係。

今天我們用 **PyTorch 手刻一個 8 層深度神經網路**，看看它能否超越傳統 ML 的表現。

**架構概覽**：
- 輸入層：HashingVectorizer 產生的 5000 維詞袋向量
- 隱藏層：8 層全連接層（Linear + ReLU 激活函數）
- 輸出層：1 個神經元輸出預測價格

> 注意：這裡的目的是建立直覺，深度學習的詳細原理將在後續課程深入探討。

In [9]:
# 準備訓練資料
# y：每個商品的真實價格（浮點數陣列）
# documents：每個商品的文字摘要（字串列表）
# 這兩者將分別進行向量化與標籤化

y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [10]:
# 文字向量化：詞袋模型（Bag of Words）
# HashingVectorizer 將每段商品描述轉換為 5000 維的稀疏向量
# - n_features=5000：雜湊空間大小（詞彙表大小的替代）
# - stop_words='english'：去除英文停用詞（如 "the", "a", "is"）
# - binary=True：只記錄詞是否出現（1/0），不記錄出現次數
# 相較於 CountVectorizer，HashingVectorizer 不需要 fit，記憶體效率更高

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

In [11]:
# 定義神經網路架構（8 層全連接網路）
# 架構：5000 → 128 → 64 → 64 → 64 → 64 → 64 → 64 → 1
# - 每層使用 ReLU 激活函數引入非線性，最後一層不加激活（直接輸出連續價格）
# - nn.Module 是 PyTorch 所有神經網路的基底類別
# - forward() 定義資料流經網路的路徑

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

In [12]:
# 資料前處理：轉換為 PyTorch Tensor 並建立 DataLoader
# 1. 將稀疏矩陣 X 轉為密集 FloatTensor（PyTorch 需要密集張量）
# 2. train_test_split：切出 1% 資料作為內部驗證集（監控訓練過程用）
# 3. TensorDataset + DataLoader：自動分批（batch_size=64）並隨機打亂順序
# 4. 初始化模型，input_size 來自向量維度（5000）

X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [13]:
# 計算模型可訓練參數數量
# 參數數量 = 每層權重矩陣 + 偏置向量的總元素數
# 這讓我們了解模型的複雜度，與後面 LLM（數十億參數）形成對比

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

Number of trainable parameters: 669,249


In [14]:
# 訓練迴圈（Training Loop）- PyTorch 深度學習的核心四步驟
#
# 損失函數：MSELoss（均方誤差），適合迴歸問題
# 優化器：Adam（自適應學習率），lr=0.001
# EPOCHS=2：對整個訓練集跑 2 輪（可依需求增加）
#
# 每個 batch 的訓練四步驟：
#   1. outputs = model(batch_X)    → 前向傳播（Forward Pass）
#   2. loss = loss_function(...)   → 計算損失
#   3. loss.backward()             → 反向傳播（Backward Pass）- 計算梯度
#   4. optimizer.step()            → 更新權重（Gradient Descent）

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [1/2], Train Loss: 8311.562, Val Loss: 12239.521


  0%|          | 0/12375 [00:00<?, ?it/s]

Epoch [2/2], Train Loss: 20430.570, Val Loss: 11108.493


In [15]:
# 推論函式：將訓練好的神經網路包裝成統一的 pricer 介面
# - model.eval()：切換為推論模式（關閉 Dropout、BatchNorm 等訓練行為）
# - torch.no_grad()：不計算梯度，節省記憶體與計算量
# - max(0, result)：確保預測價格不為負數

def neural_network(item):
    model.eval()
    with torch.no_grad():
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

In [16]:
# 評估神經網路模型的效果
# 將結果與 Day 3 的傳統 ML 模型及人類基準線比較

evaluate(neural_network, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$110 $74 $21 $175 $29 $147 $77 $36 $21 $193 $40 $225 $53 $47 $74 $15 $7 $3 $9 $37 $11 $30 $15 $82 $217 $232 $219 $27 $171 $46 $70 $155 $41 $0 $41 $347 $11 $90 $71 $61 $152 $41 $13 $56 $38 $23 $24 $16 $56 $64 $18 $64 $140 $29 $62 $85 $25 $214 $4 $42 $101 $44 $30 $35 $496 $81 $13 $340 $68 $115 $13 $26 $103 $88 $1 $31 $106 $7 $18 $62 $67 $45 $37 $54 $16 $174 $116 $141 $33 $139 $15 $61 $3 $4 $39 $44 $29 $47 $50 $159 $16 $39 $8 $81 $7 $17 $53 $302 $4 $29 $42 $14 $75 $1 $13 $170 $29 $72 $50 $85 $17 $215 $62 $43 $88 $60 $12 $86 $53 $35 $93 $27 $50 $41 $56 $18 $88 $14 $24 $5 $14 $87 $36 $97 $12 $36 $10 $160 $92 $7 $16 $13 $10 $10 $33 $94 $124 $2 $23 $20 $29 $0 $2 $22 $238 $25 $25 $34 $9 $42 $47 $10 $274 $37 $28 $14 $21 $14 $40 $34 $66 $21 $21 $66 $16 $31 $70 $66 $5 $25 $28 $9 $12 $39 $12 $41 $57 $30 $13 $4 

---

# 第三部分：前沿大型語言模型（Frontier LLMs）

## 零樣本推論（Zero-shot Inference）

前兩部分都需要用訓練資料「學習」，而 LLM 的強大之處在於：  
**不需要任何訓練，僅憑世界知識就能直接預測價格。**

我們將測試多個主流 LLM，透過相同的 `evaluate()` 函式公平比較：

| 模型 | 提供商 |
|------|--------|
| GPT-4.1 Nano | OpenAI |
| Claude Opus 4.5 | Anthropic |
| Gemini 3 Pro Preview | Google |
| Gemini 2.5 Flash Lite | Google |
| Grok 4.1 Fast | xAI |
| GPT-5.1 | OpenAI |

使用 **LiteLLM** 作為統一介面，讓所有模型的呼叫方式保持一致。  
明天（Day 5）將進一步對其中一個模型進行 **Fine-tuning**，看看訓練後能否大幅提升準確度。

In [17]:
# 建立 LLM 的提示訊息格式
# - 提示策略：直接要求模型輸出價格數字，不要附加說明
# - 回傳格式符合 OpenAI Chat API 的 messages 結構（可直接傳給 litellm）

def messages_for(item):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

In [18]:
# 查看測試集第一筆商品的描述，方便理解模型看到的輸入內容

print(test[0].summary)

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.


In [19]:
# 確認實際傳給 LLM 的訊息格式是否正確

messages_for(test[0])

[{'role': 'user',
  'content': 'Estimate the price of this product. Respond with the price, no explanation\n\nTitle: Excess V2 Distortion/Modulation Pedal  \nCategory: Music Pedals  \nBrand: Old Blood Noise  \nDescription: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  \nDetails: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.'}]

In [20]:
# GPT-4.1 Nano：OpenAI 的輕量級模型
# 透過 litellm 的 completion() 呼叫，統一所有模型的介面
# 回傳值為模型的文字輸出（應為價格數字字串）

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

In [21]:
# 測試 GPT-4.1 Nano 對第一筆商品的預測結果

gpt_4__1_nano(test[0])

'$250'

In [22]:
# 對照：查看第一筆商品的真實價格

test[0].price

219.0

In [23]:
# 正式評估 GPT-4.1 Nano 在整個測試集的表現

evaluate(gpt_4__1_nano, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $34 $25 $10 $120 $80 $24 $65 $11 $870 $363 $79 $10 $19 $29 $8 $41 $5 $40 $31 $64 $26 $65 $25 $182 $303 $705 $10 $301 $65 $40 $15 $90 $60 $35 $81 $90 $31 $36 $13 $175 $45 $22 $105 $70 $5 $27 $13 $65 $52 $20 $105 $125 $0 $97 $34 $8 $29 $52 $3 $86 $28 $46 $40 $329 $30 $90 $295 $25 $44 $17 $8 $70 $11 $5 $21 $26 $0 $8 $3 $30 $4 $0 $74 $11 $10 $32 $56 $45 $11 $3 $25 $5 $20 $2 $108 $4 $93 $20 $325 $50 $3 $11 $11 $51 $32 $10 $350 $6 $49 $10 $136 $39 $53 $54 $230 $5 $5 $64 $47 $24 $361 $80 $16 $30 $10 $5 $81 $29 $99 $179 $63 $65 $5 $85 $0 $85 $0 $93 $62 $16 $0 $30 $12 $94 $67 $5 $390 $15 $13 $3 $144 $22 $4860 $4 $129 $101 $36 $30 $5 $411 $17 $28 $2 $240 $2 $752 $30 $15 $5 $10 $3 $170 $2 $32 $101 $3 $27 $34 $23 $546 $25 $150 $99 $100 $8 $73 $7 $0 $2 $25 $99 $15 $11 $25 $70 $10 $80 $21 $11 

In [24]:
# Claude Opus 4.5：Anthropic 的旗艦模型
# 與 GPT-4.1 Nano 相比，Claude Opus 是規模更大、能力更強的模型

def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

In [25]:
# 評估 Claude Opus 4.5 的定價預測效果

evaluate(claude_opus_4_5, test)

  0%|          | 0/200 [00:00<?, ?it/s]

AuthenticationError: litellm.AuthenticationError: Missing Anthropic API Key - A call is being made to anthropic but no key is set either in the environment variables or via params. Please set `ANTHROPIC_API_KEY` in your environment vars

In [ ]:
# Gemini 3 Pro Preview：Google 的推理模型
# reasoning_effort='low'：降低推理深度以加快回應速度（推理模型預設會進行較長思考）

def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

In [ ]:
# 評估 Gemini 3 Pro Preview
# size=50：只評估 50 筆（推理模型較慢，減少筆數節省時間）
# workers=2：使用 2 個執行緒並行呼叫 API，加快評估速度

evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
# Gemini 2.5 Flash Lite：Google 的輕量快速模型
# 相較 Gemini 3 Pro，速度更快、成本更低，適合大量評估

def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-2.5-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

In [ ]:
# 評估 Gemini 2.5 Flash Lite 的定價預測效果

evaluate(gemini_2__5_flash_lite, test)

In [ ]:
# Grok 4.1 Fast：xAI（Elon Musk 旗下）的非推理快速模型
# seed=42：固定隨機種子，確保結果可重現

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
# 評估 Grok 4.1 Fast 的定價預測效果

evaluate(grok_4__1_fast, test)

In [ ]:
# GPT-5.1：OpenAI 最新旗艦推理模型
# reasoning_effort='high'：啟用高強度推理（思考時間更長，準確度更高，但較慢較貴）
# seed=42：固定隨機種子確保可重現性

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content

In [ ]:
# 評估 GPT-5.1 的定價預測效果
# 這是今天測試的最強模型，預期會有最佳表現
# 明天（Day 5）將對某個模型進行 Fine-tuning，看能否進一步超越此基準

evaluate(gpt_5__1, test)